# Batch Dataloader for Jane Street Parquet Partition 8

Memory-efficient loader for partition 8 only. Missing rows are dropped instead of filling feature NaNs with 0. No model training and no full cleaned dataset writes.

Scope:
- Target: `responder_6`
- Weight: `weight`
- Features: all `feature_` columns
- IDs preserved when present: `date_id`, `time_id`, `symbol_id`


## 1. Inspect repo and data paths

In [ ]:
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

print(f"cwd: {cwd}")
print(f"project root: {PROJECT_ROOT}")
print(f"data dir exists: {DATA_DIR.exists()} -> {DATA_DIR}")

if DATA_DIR.exists():
    print()
    print("Nearby data paths:")
    for path in sorted(DATA_DIR.glob("**/*"))[:100]:
        print(path.relative_to(PROJECT_ROOT))


In [ ]:
def resolve_partition_path(partition_id, data_dir=DATA_DIR):
    candidates = [
        data_dir / "train.parquet" / f"partition_id={partition_id}",
        data_dir / "train.parquet" / f"partition_id={partition_id}.parquet",
        data_dir / f"partition_id={partition_id}",
        data_dir / f"partition_id={partition_id}.parquet",
        data_dir / f"part_{partition_id}.parquet",
    ]
    existing = [path for path in candidates if path.exists()]
    if not existing:
        checked = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Could not find partition {partition_id}. Checked:\n{checked}")
    return existing[0]


PARTITION8_PATH = resolve_partition_path(8)
print(f"partition 8 path: {PARTITION8_PATH}")


## 2. Imports

In [ ]:
import gc

import pyarrow.dataset as ds
import pandas as pd
import numpy as np

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    pl = None
    POLARS_AVAILABLE = False

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"polars available: {POLARS_AVAILABLE}")


## 3. Inspect parquet schema only

In [ ]:
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
ID_CANDIDATES = ["date_id", "time_id", "symbol_id"]


def parquet_schema(parquet_path):
    dataset = ds.dataset(parquet_path, format="parquet")
    return dataset.schema


partition8_schema = parquet_schema(PARTITION8_PATH)
partition8_columns = partition8_schema.names

FEATURE_COLS = [col for col in partition8_columns if col.startswith("feature_")]
ID_COLS = [col for col in ID_CANDIDATES if col in partition8_columns]

print(f"partition 8 column count: {len(partition8_columns)}")
print()
print("Partition 8 columns:")
print(partition8_columns)

print()
print(f"feature column count: {len(FEATURE_COLS)}")
print(f"first 10 feature columns: {FEATURE_COLS[:10]}")
print(f"{TARGET_COL} exists: {TARGET_COL in partition8_columns}")
print(f"{WEIGHT_COL} exists: {WEIGHT_COL in partition8_columns}")
print(f"ID columns present: {ID_COLS}")

missing_required = [col for col in [TARGET_COL, WEIGHT_COL] if col not in partition8_columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")
if not FEATURE_COLS:
    raise ValueError("No feature_ columns found in partition 8 schema")


## 4. Reusable batch loader

In [ ]:
def batch_loader(
    parquet_path,
    feature_cols,
    batch_size,
    target_col=TARGET_COL,
    weight_col=WEIGHT_COL,
    id_cols=ID_CANDIDATES,
    min_date=None,
    max_date=None,
):
    """
    Stream selected parquet columns in batches, optionally filtering by date_id.

    Yields X, y, weights, ids for each cleaned batch. Rows with missing target, weight, or feature values are dropped.
    """
    parquet_path = Path(parquet_path)
    dataset = ds.dataset(parquet_path, format="parquet")
    available_cols = set(dataset.schema.names)

    selected_features = [col for col in feature_cols if col in available_cols]
    selected_ids = [col for col in id_cols if col in available_cols]
    required_cols = [target_col, weight_col]
    missing_required = [col for col in required_cols if col not in available_cols]

    if missing_required:
        raise ValueError(f"Missing required columns in {parquet_path}: {missing_required}")
    if not selected_features:
        raise ValueError(f"No requested feature columns found in {parquet_path}")
    if (min_date is not None or max_date is not None) and "date_id" not in available_cols:
        raise ValueError("date_id is required for date filtering")

    columns_to_read = selected_ids + selected_features + required_cols

    date_filter = None
    if min_date is not None:
        date_filter = ds.field("date_id") >= min_date
    if max_date is not None:
        max_filter = ds.field("date_id") <= max_date
        date_filter = max_filter if date_filter is None else date_filter & max_filter

    scanner = dataset.scanner(
        columns=columns_to_read,
        filter=date_filter,
        batch_size=batch_size,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    for record_batch in scanner.to_batches():
        batch_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
        batch_df = batch_df.dropna(subset=selected_features + [target_col, weight_col])
        if batch_df.empty:
            continue

        ids = batch_df[selected_ids].reset_index(drop=True) if selected_ids else pd.DataFrame(index=range(len(batch_df)))
        X = batch_df[selected_features].to_numpy(dtype=np.float32, copy=False)
        y = batch_df[target_col].to_numpy(dtype=np.float32, copy=False)
        weights = batch_df[weight_col].to_numpy(dtype=np.float32, copy=False)

        yield X, y, weights, ids


## 5. Test one small batch from partition 8

In [ ]:
BATCH_SIZE = 2_048

loader = batch_loader(PARTITION8_PATH, FEATURE_COLS, batch_size=BATCH_SIZE)
X, y, weights, ids = next(loader)

partition8_batch_shape = X.shape

print(f"X shape:       {X.shape}")
print(f"y shape:       {y.shape}")
print(f"weights shape: {weights.shape}")
print(f"ids shape:     {ids.shape}")
print()
print("First few ID rows:")
display(ids.head())
print()
print("Missing value counts after cleaning:")
print({
    "target_missing_after": int(pd.isna(y).sum()),
    "weight_missing_after": int(pd.isna(weights).sum()),
    "feature_missing_after": int(np.isnan(X).sum()),
})

del X, y, weights, ids, loader
gc.collect()


## 6. Collect limited batches

In [ ]:
def collect_batches(loader, max_rows):
    """Collect at most max_rows from a batch loader into memory."""
    X_parts = []
    y_parts = []
    weight_parts = []
    rows_collected = 0

    for X, y, weights, ids in loader:
        remaining = max_rows - rows_collected
        if remaining <= 0:
            break

        take = min(len(y), remaining)
        X_parts.append(X[:take])
        y_parts.append(y[:take])
        weight_parts.append(weights[:take])
        rows_collected += take

        del X, y, weights, ids
        gc.collect()

        if rows_collected >= max_rows:
            break

    if not X_parts:
        return (
            np.empty((0, len(FEATURE_COLS)), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    return (
        np.concatenate(X_parts, axis=0),
        np.concatenate(y_parts, axis=0),
        np.concatenate(weight_parts, axis=0),
    )


In [ ]:
LOCAL_DEMO_MAX_ROWS = 5_000

small_loader = batch_loader(PARTITION8_PATH, FEATURE_COLS, batch_size=BATCH_SIZE)
X_small, y_small, weights_small = collect_batches(small_loader, max_rows=LOCAL_DEMO_MAX_ROWS)

print(f"Collected rows: {len(y_small):,}")
print(f"X_small shape:       {X_small.shape}")
print(f"y_small shape:       {y_small.shape}")
print(f"weights_small shape: {weights_small.shape}")
print(f"X missing values: {int(np.isnan(X_small).sum())}")
print(f"y missing values: {int(np.isnan(y_small).sum())}")
print(f"weight missing values: {int(np.isnan(weights_small).sum())}")

del X_small, y_small, weights_small, small_loader
gc.collect()


## 7. Optional date filter within partition 8

In [ ]:
def date_id_range(parquet_path, batch_size=BATCH_SIZE):
    """Find min/max date_id by streaming only the date_id column."""
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=["date_id"],
        batch_size=batch_size,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    min_seen = None
    max_seen = None
    for record_batch in scanner.to_batches():
        values = record_batch.column(0).to_numpy(zero_copy_only=False)
        values = values[~pd.isna(values)]
        if len(values) == 0:
            continue
        batch_min = int(values.min())
        batch_max = int(values.max())
        min_seen = batch_min if min_seen is None else min(min_seen, batch_min)
        max_seen = batch_max if max_seen is None else max(max_seen, batch_max)

    return min_seen, max_seen


partition8_min_date, partition8_max_date = date_id_range(PARTITION8_PATH)
print(f"partition 8 date_id range: {partition8_min_date} to {partition8_max_date}")


In [ ]:
single_date_loader = batch_loader(
    PARTITION8_PATH,
    FEATURE_COLS,
    batch_size=BATCH_SIZE,
    min_date=partition8_max_date,
    max_date=partition8_max_date,
)
X_date, y_date, weights_date, ids_date = next(single_date_loader)

print(f"single-date sample date_id: {partition8_max_date}")
print(f"X shape:       {X_date.shape}")
print(f"y shape:       {y_date.shape}")
print(f"weights shape: {weights_date.shape}")
print(f"ids date range: {int(ids_date['date_id'].min())} to {int(ids_date['date_id'].max())}")

del X_date, y_date, weights_date, ids_date, single_date_loader
gc.collect()


## Summary

In [ ]:
print(f"notebook path: {PROJECT_ROOT / 'notebooks' / '01_loader_p8.ipynb'}")
print(f"detected partition 8 parquet path: {PARTITION8_PATH}")
print(f"number of features: {len(FEATURE_COLS)}")
print(f"one-batch shape for partition 8: {partition8_batch_shape}")
